# Pregunta 2 — Análisis exploratorio de Sen1Floods11

Antes de entrenar nada conviene contestar cuatro preguntas sobre los
datos, porque cada una condiciona una decisión del modelo:

1. **¿Cuánto hay, y de dónde?** — define qué significa "generalizar".
2. **¿El agua se separa de la tierra solo mirando los decibeles?** — si
   se separa bien, un umbral alcanza y la red tiene poco que agregar.
3. **¿Las regiones se parecen entre sí?** — si no, el salto a una región
   nueva no es un detalle sino el problema central.
4. **¿Qué casos raros hay?** — son los que después aparecen como fallos.

Todas las celdas **solo leen** datos de disco. Ninguna entrena nada.

In [ ]:
import json
import pathlib
import sys

import numpy as np
import rasterio
import matplotlib.pyplot as plt

# Raiz del proyecto en la maquina local. Orden: la variable REDES_RAIZ, la
# carpeta actual o alguna superior, y por ultimo la ruta fija de esta PC. Asi
# funciona en Jupyter/VS Code y en Colab conectado a un entorno de ejecucion
# local, aunque Jupyter se haya lanzado desde otra carpeta.
import os
RAIZ_LOCAL = pathlib.Path("D:/Backend/redesneuronales")
RAIZ = pathlib.Path(os.environ.get("REDES_RAIZ", pathlib.Path.cwd()))
while not (RAIZ / "pregunta1").is_dir() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
if not (RAIZ / "pregunta1").is_dir():
    RAIZ = RAIZ_LOCAL
if not (RAIZ / "pregunta1").is_dir():
    raise FileNotFoundError(
        f"No encuentro el proyecto (busque desde {pathlib.Path.cwd()} y en "
        f"{RAIZ_LOCAL}). En Colab: Conectar -> 'Conectar a un entorno de "
        "ejecucion local', o define REDES_RAIZ con la ruta del proyecto.")
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

# Las rutas salen de pregunta2/rutas.py: prefieren pregunta2/datos/
# (los enlaces de alinear_datos.py) y caen a data/ si no existen.
from pregunta2.rutas import MANO, SPLITS, describir
print(" ", describir())
IGNORAR = -1

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 9})


def chips(split):
    ruta = SPLITS / f"flood_{split}_data.csv"
    filas = ruta.read_text(encoding="utf-8").strip().splitlines()
    return [f.split(",")[0].replace("_S1Hand.tif", "") for f in filas if f.strip()]


def cargar(chip):
    """(2, 512, 512) en dB y la etiqueta (512, 512) con 1/0/-1."""
    with rasterio.open(MANO / "S1Hand" / f"{chip}_S1Hand.tif") as r:
        x = r.read().astype(np.float32)
    with rasterio.open(MANO / "LabelHand" / f"{chip}_LabelHand.tif") as r:
        y = r.read(1)
    return x, y


print("chips disponibles:", {s: len(chips(s))
                             for s in ("train", "valid", "test", "bolivia")})

## 1. Qué hay, y cómo está repartido

El dataset trae su propia partición oficial, y esa partición ya contiene
una decisión de diseño importante.

In [ ]:
import collections

print(f"{'particion':<10} {'chips':>6}  regiones")
print("-" * 72)
for s in ("train", "valid", "test", "bolivia"):
    c = chips(s)
    regs = collections.Counter(x.split("_")[0] for x in c)
    print(f"{s:<10} {len(c):>6}  {dict(sorted(regs.items()))}")

print("\nBolivia esta SOLA en su particion y no aparece en ninguna otra:")
print("el dataset la retiene entera como region de generalizacion.")
print("Eso no es un detalle del reparto, es el experimento del entregable 4.")

## 2. ¿El agua se separa solo con los decibeles?

Esta es la pregunta que decide si el problema es fácil o difícil. El agua
calma actúa como un espejo: refleja la señal del radar lejos de la antena
y vuelve poca energía, así que **debería** verse oscura.

Lo medimos sobre una muestra de chips de entrenamiento.

In [ ]:
rng = np.random.default_rng(0)
muestra = rng.choice(chips("train"), 40, replace=False)

vv_agua, vv_tierra, vh_agua, vh_tierra = [], [], [], []
for ch in muestra:
    x, y = cargar(ch)
    fin = np.isfinite(x).all(axis=0)
    agua = (y == 1) & fin
    tierra = (y == 0) & fin
    # Submuestreo: 40 chips completos serian 10 M de pixeles por clase.
    for mascara, dv, dh in ((agua, vv_agua, vh_agua),
                            (tierra, vv_tierra, vh_tierra)):
        idx = np.flatnonzero(mascara.ravel())
        if idx.size:
            sel = rng.choice(idx, min(4000, idx.size), replace=False)
            dv.append(x[0].ravel()[sel])
            dh.append(x[1].ravel()[sel])

vv_agua = np.concatenate(vv_agua); vv_tierra = np.concatenate(vv_tierra)
vh_agua = np.concatenate(vh_agua); vh_tierra = np.concatenate(vh_tierra)
print(f"muestra: {len(vv_agua):,} pixeles de agua, {len(vv_tierra):,} de tierra")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, (a, t, nombre) in zip(axes, ((vv_agua, vv_tierra, "VV"),
                                     (vh_agua, vh_tierra, "VH"))):
    ax.hist(t, bins=120, range=(-35, 5), density=True, alpha=0.6,
            label="no agua", color="#8a6d3b")
    ax.hist(a, bins=120, range=(-35, 5), density=True, alpha=0.6,
            label="agua", color="#2a7fbf")
    ax.set_xlabel(f"{nombre} (dB)"); ax.set_ylabel("densidad")
    ax.set_title(f"{nombre}: agua contra tierra"); ax.legend()
plt.tight_layout(); plt.show()

print(f"\nmediana VV: agua {np.median(vv_agua):+.1f} dB | "
      f"tierra {np.median(vv_tierra):+.1f} dB | "
      f"separacion {np.median(vv_tierra)-np.median(vv_agua):.1f} dB")
print(f"mediana VH: agua {np.median(vh_agua):+.1f} dB | "
      f"tierra {np.median(vh_tierra):+.1f} dB | "
      f"separacion {np.median(vh_tierra)-np.median(vh_agua):.1f} dB")

### Lo que hay que mirar en esos histogramas

Las dos clases están **separadas pero se solapan**. Esa es exactamente la
razón por la que una umbralización como Otsu funciona razonablemente
—hay un valle entre los dos modos— y también por la que no basta: la zona
de solape son píxeles que ningún umbral global puede resolver, y es ahí
donde una red que mira el **contexto espacial** puede aportar.

Si las distribuciones estuvieran perfectamente separadas, la U-Net sería
un lujo innecesario. Si estuvieran completamente superpuestas, ningún
modelo de un solo píxel funcionaría.

In [ ]:
# Cuanto se solapan, en numeros: que tan bien separa el MEJOR umbral global
# posible sobre VV. Es el techo de cualquier metodo que mire un pixel solo.
bordes = np.linspace(-35, 5, 400)
mejor = (0.0, None)
for u in bordes:
    # Agua = por debajo del umbral.
    vp = (vv_agua <= u).sum(); fn = (vv_agua > u).sum()
    fp = (vv_tierra <= u).sum()
    iou = vp / max(1, vp + fp + fn)
    if iou > mejor[0]:
        mejor = (iou, u)
print(f"mejor umbral global sobre VV: {mejor[1]:.1f} dB -> IoU {mejor[0]:.4f}")
print("\n  Ese es el TECHO de un metodo que mira un pixel aislado, con el")
print("  umbral elegido con la respuesta en la mano (o sea, optimista).")
print("  Todo lo que una U-Net gane por encima de eso viene del contexto")
print("  espacial: la forma de la mancha, los bordes, la textura.")

## 3. ¿Las regiones se parecen entre sí?

Acá está el corazón del entregable 4. Si cada región tiene su propia
distribución de retrodispersión, un modelo que aprende de diez no tiene
por qué funcionar en la undécima.

In [ ]:
import collections

por_region = collections.defaultdict(lambda: {"vv": [], "agua": 0, "obs": 0,
                                              "chips": 0})
todos = [(s, c) for s in ("train", "valid", "test", "bolivia")
         for c in chips(s)]
rng = np.random.default_rng(1)
for s, ch in todos:
    reg = ch.split("_")[0]
    d = por_region[reg]
    if d["chips"] >= 12:          # tope por region, para que no tarde
        continue
    x, y = cargar(ch)
    fin = np.isfinite(x[0])
    obs = (y != IGNORAR) & fin
    if obs.sum() < 1000:
        continue
    idx = np.flatnonzero(obs.ravel())
    sel = rng.choice(idx, min(3000, idx.size), replace=False)
    d["vv"].append(x[0].ravel()[sel])
    d["agua"] += int((y == 1).sum()); d["obs"] += int(obs.sum())
    d["chips"] += 1

print(f"{'region':<12} {'chips':>6} {'% agua':>8} {'VV p25':>8} "
      f"{'VV med':>8} {'VV p75':>8}")
print("-" * 56)
filas = []
for reg, d in sorted(por_region.items()):
    v = np.concatenate(d["vv"])
    q = np.percentile(v, [25, 50, 75])
    filas.append((reg, 100*d["agua"]/max(1, d["obs"]), q))
    print(f"{reg:<12} {d['chips']:>6} {100*d['agua']/max(1,d['obs']):>7.2f}% "
          f"{q[0]:>8.1f} {q[1]:>8.1f} {q[2]:>8.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

regiones = sorted(por_region)
datos = [np.concatenate(por_region[r]["vv"]) for r in regiones]
# tick_labels y no labels: matplotlib 3.9 renombro el argumento.
axes[0].boxplot(datos, tick_labels=regiones, showfliers=False)
axes[0].set_ylabel("VV (dB)")
axes[0].set_title("Retrodispersion VV por region (pixeles observados)")
axes[0].tick_params(axis="x", rotation=55)
# Bolivia se resalta: es la region que el modelo NUNCA ve.
if "Bolivia" in regiones:
    i = regiones.index("Bolivia") + 1
    axes[0].axvspan(i - 0.4, i + 0.4, color="crimson", alpha=0.12)

prev = [f[1] for f in sorted(filas)]
nombres = [f[0] for f in sorted(filas)]
colores = ["crimson" if n == "Bolivia" else "#4c78a8" for n in nombres]
axes[1].bar(nombres, prev, color=colores)
axes[1].set_ylabel("% de agua entre lo observado")
axes[1].set_title("Prevalencia de agua por region")
axes[1].tick_params(axis="x", rotation=55)
plt.tight_layout(); plt.show()

print("La franja roja es Bolivia: la region que el modelo nunca ve durante")
print("el entrenamiento. Si su caja se parece a las demas, el problema no")
print("es el aspecto del radar sino otra cosa (prevalencia, tipo de agua).")

## 4. ¿Cuánto se puede supervisar, en realidad?

El `-1` no es un detalle de formato: define qué fracción de cada chip
puede entrar en la pérdida.

In [ ]:
frac_nodata, frac_agua = [], []
rng = np.random.default_rng(2)
for ch in rng.choice(chips("train"), 80, replace=False):
    with rasterio.open(MANO / "LabelHand" / f"{ch}_LabelHand.tif") as r:
        y = r.read(1)
    frac_nodata.append((y == IGNORAR).mean())
    obs = (y != IGNORAR).sum()
    frac_agua.append((y == 1).sum() / max(1, obs))

frac_nodata = np.array(frac_nodata); frac_agua = np.array(frac_agua)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].hist(100*frac_nodata, bins=25, color="#999")
axes[0].set_xlabel("% del chip sin dato (-1)"); axes[0].set_ylabel("chips")
axes[0].set_title("Cuanto de cada chip NO se puede supervisar")
axes[1].hist(100*frac_agua, bins=25, color="#2a7fbf")
axes[1].set_xlabel("% de agua entre lo observado"); axes[1].set_ylabel("chips")
axes[1].set_title("Prevalencia de agua por chip")
plt.tight_layout(); plt.show()

print(f"sin dato: mediana {100*np.median(frac_nodata):.1f}% | "
      f"maximo {100*frac_nodata.max():.1f}%")
print(f"  chips con mas del 50% sin dato: {(frac_nodata > 0.5).sum()} de 80")
print(f"\nagua: mediana {100*np.median(frac_agua):.1f}% | "
      f"chips con menos del 1% de agua: {(frac_agua < 0.01).sum()} de 80")
print("\n  Las dos colas importan. Los chips con mucho -1 aportan poca")
print("  supervision (y romperian la perdida si no se enmascarara);")
print("  los chips casi secos son los que hacen inestable la IoU por chip.")

## 5. Los casos raros, que después son los fallos

Buscamos chips donde el agua **no** se ve oscura. Son los que el entregable
5 pide analizar: vegetación inundada, zona urbana inundada, sombra de radar.

In [ ]:
# Para cada chip de prueba, cuanto mas oscura es el agua que la tierra.
# Si la diferencia es chica o negativa, el chip es anomalo.
contrastes = []
for ch in chips("test"):
    x, y = cargar(ch)
    fin = np.isfinite(x[0])
    a, t = (y == 1) & fin, (y == 0) & fin
    if a.sum() < 2000 or t.sum() < 2000:
        continue
    contrastes.append((float(np.median(x[0][t]) - np.median(x[0][a])), ch))
contrastes.sort()

print("chips donde el agua se ve MENOS oscura de lo esperado (o mas brillante):")
for d, ch in contrastes[:5]:
    print(f"  {ch:<26} tierra - agua = {d:+6.1f} dB")
print("\nchips con el contraste tipico:")
for d, ch in contrastes[-3:]:
    print(f"  {ch:<26} tierra - agua = {d:+6.1f} dB")

In [ ]:
def pintar(ch, ejes):
    x, y = cargar(ch)
    def estirar(v):
        v = np.where(np.isfinite(v), v, np.nan)
        lo, hi = np.nanpercentile(v, [2, 98])
        return np.nan_to_num(np.clip((v - lo) / max(hi - lo, 1e-6), 0, 1))
    rgb = np.stack([estirar(x[0]), estirar(x[1]),
                    estirar(x[0] - x[1])], axis=-1)
    col = np.zeros((*y.shape, 3))
    col[y == 1] = (0.16, 0.5, 1.0)
    col[y == 0] = (0.2, 0.24, 0.29)
    col[y == IGNORAR] = (0.84, 0.84, 0.84)
    ejes[0].imshow(rgb); ejes[0].set_title(ch, fontsize=8)
    ejes[1].imshow(col); ejes[1].set_title("etiqueta", fontsize=8)
    for e in ejes:
        e.axis("off"); e.grid(False)

casos = [contrastes[0][1], contrastes[1][1], contrastes[-1][1]]
fig, axes = plt.subplots(len(casos), 2, figsize=(7.5, 3.8*len(casos)))
for fila, ch in enumerate(casos):
    pintar(ch, axes[fila])
plt.suptitle("Arriba: agua que NO se ve oscura.  Abajo: el caso tipico",
             fontsize=10)
plt.tight_layout(); plt.show()

print("R = VV, G = VH, B = VV - VH.  En la etiqueta: azul agua,")
print("gris oscuro no agua, gris claro SIN DATO.")

### Por qué pasa eso

Cuando el agua cubre vegetación o estructuras, la señal rebota **dos
veces** —del suelo inundado al tronco o a la pared, y de ahí a la antena—
y vuelve **más** energía que desde el suelo seco. El agua inundando un
bosque o una ciudad puede verse más brillante que la tierra que la rodea,
que es exactamente lo contrario de lo que un umbral asume.

Es el modo de fallo más importante del entregable 5, y acá está medido en
decibeles en vez de descrito.

## 6. ¿Qué tan correlacionadas están VV y VH?

Si las dos bandas dijeran lo mismo, la segunda sería peso muerto.

In [ ]:
r_agua = np.corrcoef(vv_agua, vh_agua)[0, 1]
r_tierra = np.corrcoef(vv_tierra, vh_tierra)[0, 1]
print(f"correlacion VV-VH en agua  : {r_agua:+.3f}")
print(f"correlacion VV-VH en tierra: {r_tierra:+.3f}")

fig, ax = plt.subplots(figsize=(5, 4.4))
paso = max(1, len(vv_tierra) // 6000)
ax.scatter(vv_tierra[::paso], vh_tierra[::paso], s=1, alpha=0.18,
           label="no agua", color="#8a6d3b")
paso = max(1, len(vv_agua) // 6000)
ax.scatter(vv_agua[::paso], vh_agua[::paso], s=1, alpha=0.18,
           label="agua", color="#2a7fbf")
ax.set_xlabel("VV (dB)"); ax.set_ylabel("VH (dB)")
ax.set_title("Las dos polarizaciones")
leg = ax.legend(markerscale=8)
plt.tight_layout(); plt.show()

print("\nLas dos nubes se separan mejor en el plano que en cualquiera de")
print("los dos ejes por separado: la segunda banda SI aporta. Es la razon")
print("por la que la entrada de la red son 2 canales y no 1.")

# Lo que este análisis deja decidido

| hallazgo | consecuencia en el modelo |
|---|---|
| VV y VH están en decibeles, negativos, con mínimos de −79 dB | normalizar con estadísticas **medidas** y recortar por percentiles, no con constantes de ImageNet |
| ~13% de los píxeles son `-1` | la pérdida **tiene que** enmascarar |
| el agua es ~9.5% de lo observado | la métrica es IoU, no exactitud; y la pérdida combina BCE con Dice |
| las clases se separan pero se solapan | un umbral global llega hasta cierto punto; el resto lo tiene que poner el contexto espacial |
| las regiones tienen distribuciones distintas | generalizar a una región nueva es el problema, no un detalle |
| hay agua que se ve **brillante** | el modo de fallo del entregable 5, cuantificado |
| VV y VH no son redundantes | la entrada son 2 canales |